
# Training PPO Agents on a Planar Muscle-Tendon Arm Environment using MJX

This notebook adapts a standard MuJoCo planar arm model with muscle-tendon actuators to run on the GPU via **MJX** and **Brax**. It implements an operational space reach task, tracking a randomized coordinate point in a 2D workspace.

**A Colab runtime with GPU acceleration is required.** If you're using a CPU-only runtime, please switch via **Runtime > Change runtime type**.

---


In [ ]:
#@title Install pre-requisites
!pip install mujoco
!pip install mujoco_mjx
!pip install brax
!pip install playground

In [ ]:

# @title Check if MuJoCo installation was successful

import distutils.util
import os
import subprocess

if subprocess.run('nvidia-smi').returncode:
raise RuntimeError(
'Cannot communicate with GPU. '
'Make sure you are using a GPU Colab runtime. '
'Go to the Runtime menu and select Choose runtime type.'
)

NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
f.write("""{
"file_format_version" : "1.0.0",
"ICD" : {
"library_path" : "libEGL_nvidia.so.0"
}
}
""")

print('Setting environment variable to use GPU rendering:')
%env MUJOCO_GL=egl

try:
    print('Checking that the installation succeeded:')
    import mujoco
    mujoco.MjModel.from_xml_string('')
except Exception as e:
    raise e from RuntimeError(
    'Something went wrong during installation. Check the shell output above '
    'for more information.'
    )

print('Installation successful.')

# Tell XLA to use Triton GEMM, improving steps/sec on GPUs

xla_flags = os.environ.get('XLA_FLAGS', '')
xla_flags += ' --xla_gpu_triton_gemm_any=True'
os.environ['XLA_FLAGS'] = xla_flags

In [ ]:

# @title Import packages for plotting and creating graphics

import itertools
import time
from typing import Callable, List, NamedTuple, Optional, Union, Any, Dict
import numpy as np

print("Installing mediapy:")
!command -v ffmpeg >/dev/null || (apt update && apt install -y ffmpeg)
!pip install -q mediapy
import mediapy as media
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True, linewidth=100)

In [ ]:

# @title Import MuJoCo, MJX, and Brax

from datetime import datetime
import functools
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.agents.ppo import train as ppo
from IPython.display import clear_output
import jax
from jax import numpy as jp
from ml_collections import config_dict
from mujoco import mjx
from mujoco_playground._src.dm_control_suite.cartpole import reward
from mujoco_playground._src.dm_control_suite import mjx_env


## Environment Definition

We define our inline MuJoCo XML containing a 2-link planar arm modeled with antagonist muscle pairs matching the configuration principles of your original simulation.

The task objective is to minimize distance to a target coordinate within the arm's workspace. The tracking penalty criteria uses an exponential function relative to the target vector:

$$R = e^{-\beta \|x_{\text{tip}} - x_{\text{target}}\|^2} - \alpha \|a\|^2$$

---


In [ ]:
ARM_XML = """




```
<worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <body name="base" pos="0 0 0">
        <geom type="cylinder" size="0.1 0.02" rgba="0.5 0.5 0.5 1"/>
        <body name="upper_arm" pos="0 0 0">
            <joint name="shoulder" type="hinge" axis="0 0 1" range="-90 90"/>
            <geom type="capsule" fromto="0 0 0 0.3 0 0" size="0.03" rgba="0.9 0.4 0.4 1"/>
            <body name="forearm" pos="0.3 0 0">
                <joint name="elbow" type="hinge" axis="0 0 1" range="-150 150"/>
                <geom type="capsule" fromto="0 0 0 0.25 0 0" size="0.025" rgba="0.4 0.9 0.4 1"/>
                <body name="tip" pos="0.25 0 0">
                    <site name="tip_site" pos="0 0 0" size="0.01" rgba="0 0 1 1"/>
                </body>
            </body>
        </body>
    </body>
    <site name="target" pos="0.3 0.2 0" size="0.02" rgba="1 0 0 0.6"/>
</worldbody>

<tendon>
    <fixed name="t_shoulder_pull"><joint joint="shoulder" coef="1"/></fixed>
    <fixed name="t_shoulder_push"><joint joint="shoulder" coef="-1"/></fixed>
    <fixed name="t_elbow_pull"><joint joint="elbow" coef="1"/></fixed>
    <fixed name="t_elbow_push"><joint joint="elbow" coef="-1"/></fixed>
</tendon>

<actuator>
    <muscle name="m_shoulder_pull" tendon="t_shoulder_pull" scale="200"/>
    <muscle name="m_shoulder_push" tendon="t_shoulder_push" scale="200"/>
    <muscle name="m_elbow_pull" tendon="t_elbow_pull" scale="200"/>
    <muscle name="m_elbow_push" tendon="t_elbow_push" scale="200"/>
</actuator>

```

class PlanarArmTendon(mjx_env.MjxEnv):
"""Planar muscle-tendon arm tracking environment running fully on MJX."""

def **init**(self, config: config_dict.ConfigDict = None):
if config is None:
config = config_dict.ConfigDict()
config.sim_dt = 0.004
config.ctrl_dt = 0.02
config.episode_length = 200

```
super().__init__(config)
self._mj_model = mujoco.MjModel.from_xml_string(ARM_XML)
self._mj_model.opt.timestep = self.sim_dt
self._mjx_model = mjx.put_model(self._mj_model)
self._post_init()

```

def _post_init(self) -> None:
self._shoulder_qposadr = self._mj_model.joint("shoulder").id
self._elbow_qposadr = self._mj_model.joint("elbow").id
self._tip_body_id = self._mj_model.body("tip").id

def reset(self, rng: jax.Array) -> mjx_env.State:
rng, rng_q, rng_target = jax.random.split(rng, 3)

```
# Initialize positions with slight variation
qpos = jp.zeros(self.mjx_model.nq)
qpos = qpos.at[self._shoulder_qposadr].set(jax.random.uniform(rng_q, (), minval=-0.2, maxval=0.2))
qpos = qpos.at[self._elbow_qposadr].set(jax.random.uniform(rng_q, (), minval=-0.2, maxval=0.2))
qvel = jp.zeros(self.mjx_model.nv)

# Generate a random target within operational workspace radius
radius = jax.random.uniform(rng_target, (), minval=0.2, maxval=0.5)
angle = jax.random.uniform(rng_target, (), minval=-jp.pi/2, maxval=jp.pi/2)
target_pos = jp.array([radius * jp.cos(angle), radius * jp.sin(angle)])

data = mjx_env.init(self.mjx_model, qpos=qpos, qvel=qvel)

metrics = {
    "reward/tracking": jp.zeros(()),
    "reward/control_penalty": jp.zeros(()),
    "distance": jp.zeros(())
}

info = {"rng": rng, "target": target_pos}
obs = self._get_obs(data, info)
reward_val, done = jp.zeros(2)

return mjx_env.State(data, obs, reward_val, done, metrics, info)

```

def step(self, state: mjx_env.State, action: jax.Array) -> mjx_env.State:
# Muscle inputs are bound between 0 and 1
ctrl = jp.clip(action, 0.0, 1.0)
data = mjx_env.step(self.mjx_model, state.data, ctrl, self.n_substeps)

```
reward_val = self._get_reward(data, ctrl, state.info, state.metrics)
obs = self._get_obs(data, state.info)

done = jp.isnan(data.qpos).any() | jp.isnan(data.qvel).any()
done = done.astype(float)

return mjx_env.State(data, obs, reward_val, done, state.metrics, state.info)

```

def _get_obs(self, data: mjx.Data, info: dict[str, Any]) -> jax.Array:
tip_pos = data.xpos[self._tip_body_id][:2]
target_pos = info["target"]

```
return jp.concatenate([
    jp.sin(data.qpos),
    jp.cos(data.qpos),
    data.qvel,
    tip_pos,
    target_pos,
    target_pos - tip_pos
])

```

def _get_reward(self, data: mjx.Data, action: jax.Array, info: dict[str, Any], metrics: dict[str, Any]) -> jax.Array:
tip_pos = data.xpos[self._tip_body_id][:2]
target_pos = info["target"]

```
dist = jp.linalg.norm(tip_pos - target_pos)
metrics["distance"] = dist

# Exponential reward landscape for precise tracking
tracking = jp.exp(-5.0 * dist)
metrics["reward/tracking"] = tracking

control_penalty = -0.05 * jp.sum(jp.square(action))
metrics["reward/control_penalty"] = control_penalty

return tracking + control_penalty

```

@property
def xml_path(self) -> str:
return ""

@property
def action_size(self) -> int:
return self.mjx_model.nu

@property
def mj_model(self) -> mujoco.MjModel:
return self._mj_model

@property
def mjx_model(self) -> mjx.Model:
return self._mjx_model


## Unrolled Random Step Testing

Let's quickly check the functionality of our compiled JAX environment with an initialization check and an unrolled loop executing random actions.

---


In [ ]:
env = PlanarArmTendon()
jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)

state = jit_reset(jax.random.PRNGKey(42))
print("Environment initialized successfully.")
print("Observations shape:", state.obs.shape)
print("Observation Allocation device:", state.obs.device)


## Training the Agent with PPO

We set up a configuration dictionary defining our hyper-parameters and call `brax.training.agents.ppo` to train our muscle activation controller network directly inside the GPU.

---


In [ ]:

# Setup local hyperparameters dictionary

ppo_params = {
"num_timesteps": 10_000_000,
"num_envs": 2048,
"unroll_length": 10,
"num_minibatches": 32,
"num_updates_per_batch": 4,
"discounting": 0.97,
"learning_rate": 3e-4,
"entropy_cost": 1e-3,
"num_evals": 20,
"seed": 0
}

x_data, y_data, y_dataerr = [], [], []
times = [datetime.now()]

def progress(num_steps, metrics):
clear_output(wait=True)
times.append(datetime.now())
x_data.append(num_steps)
y_data.append(metrics["eval/episode_reward"])
y_dataerr.append(metrics["eval/episode_reward_std"])

plt.figure(figsize=(10, 5))
plt.xlim([0, ppo_params["num_timesteps"] * 1.1])
plt.xlabel("# Environment Steps")
plt.ylabel("Reward per Episode")
plt.title(f"Step: {num_steps} -> Mean Evaluation Reward: {y_data[-1]:.3f}")
plt.errorbar(x_data, y_data, yerr=y_dataerr, color="orange")
display(plt.gcf())
plt.close()

# Prepare Network Factories

network_factory = ppo_networks.make_ppo_networks

train_fn = functools.partial(
ppo.train,
ppo_params,
network_factory=network_factory,
progress_fn=progress
)

from mujoco_playground import wrapper
make_inference_fn, params, metrics = train_fn(
environment=env,
wrap_env_fn=wrapper.wrap_for_brax_training,
)

print(f"Time to compile (JIT): {times[1] - times[0]}")
print(f"Time to complete training: {times[-1] - times[1]}")


## Interactive Policy Rollout Visualization

With our network optimizations fully initialized, we evaluate the system deterministic output and save the kinematic behavior directly into a video playback sequence.

---


In [ ]:
jit_inference_fn = jax.jit(make_inference_fn(params, deterministic=True))

rng = jax.random.PRNGKey(123)
state = jit_reset(rng)
rollout = [state]

# Execute evaluation episode loop

for i in range(env.config.episode_length):
act_rng, rng = jax.random.split(rng)
ctrl, _ = jit_inference_fn(state.obs, act_rng)
state = jit_step(state, ctrl)
rollout.append(state)

# Render frames across simulated sequence steps

print("Rendering rollout frames via MuJoCo rendering pipelines...")
frames = env.render(rollout)
media.show_video(frames, fps=1.0 / env.dt)

```

```